# Setup Instructions

This notebook includes the setup instructions for LaneSegNet as described in the README file. Follow these steps to prepare the environment and dataset for training.

In [ ]:
# Step 0: Clone the Repository
!git clone https://github.com/OpenDriveLab/LaneSegNet.git
%cd /Users/vishwajeethogale/Desktop/Research/LaneSegNet

In [ ]:
# Step 1: Create a Conda Environment
!conda create -n lanesegnet python=3.8 -y
!conda activate lanesegnet

# Step 2: Install CUDA Toolkit (Optional)
!conda install cudatoolkit=11.1.1 -c conda-forge

# Step 3: Install PyTorch
!pip install torch==1.9.1+cu111 torchvision==0.10.1+cu111 -f https://download.pytorch.org/whl/torch_stable.html

# Step 4: Install mm-series Packages
!pip install mmcv-full==1.5.2 -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.9.0/index.html
!pip install mmdet==2.26.0
!pip install mmsegmentation==0.29.1
!pip install mmdet3d==1.0.0rc6

# Step 5: Install Other Required Packages
!pip install -r requirements.txt

# Step 6: Prepare Dataset
!mkdir -p data
!ln -s {Path to OpenLane-V2 repo}/data/OpenLane-V2 ./data/
!python ./tools/data_process.py

In [ ]:
# Import necessary libraries
from __future__ import division
import argparse
import copy
import os
import time
import warnings
from os import path as osp

import mmcv
import torch
import torch.distributed as dist
from mmcv import Config, DictAction
from mmcv.runner import get_dist_info, init_dist

from mmdet import __version__ as mmdet_version
from mmdet3d import __version__ as mmdet3d_version
from mmdet3d.apis import init_random_seed, train_model
from mmdet3d.datasets import build_dataset
from mmdet3d.models import build_model
from mmdet3d.utils import collect_env, get_root_logger
from mmdet.apis import set_random_seed
from mmseg import __version__ as mmseg_version

try:
    from mmdet.utils import setup_multi_processes
except ImportError:
    from mmdet3d.utils import setup_multi_processes

# Define the training function
def train_lanesegnet(config_path, work_dir=None, resume_from=None, auto_resume=False, no_validate=False, gpu_id=0, seed=42, diff_seed=False, deterministic=False, autoscale_lr=False):
    cfg = Config.fromfile(config_path)
    
    # Set multi-process settings
    setup_multi_processes(cfg)

    # Set cudnn_benchmark
    if cfg.get('cudnn_benchmark', False):
        torch.backends.cudnn.benchmark = True

    # Work directory setup
    if work_dir is not None:
        cfg.work_dir = work_dir
    elif cfg.get('work_dir', None) is None:
        cfg.work_dir = osp.join('./work_dirs', osp.splitext(osp.basename(config_path))[0])
    if resume_from is not None:
        cfg.resume_from = resume_from
    if auto_resume:
        cfg.auto_resume = auto_resume

    cfg.gpu_ids = [gpu_id]

    if autoscale_lr:
        if 'auto_scale_lr' in cfg and 'base_batch_size' in cfg.auto_scale_lr:
            cfg.auto_scale_lr.enable = True

    # Initialize distributed environment
    distributed = False
    
    # Create work directory
    mmcv.mkdir_or_exist(osp.abspath(cfg.work_dir))
    cfg.dump(osp.join(cfg.work_dir, osp.basename(config_path)))
    
    # Initialize logger
    timestamp = time.strftime('%Y%m%d_%H%M%S', time.localtime())
    log_file = osp.join(cfg.work_dir, f'{timestamp}.log')
    logger = get_root_logger(log_file=log_file, log_level=cfg.log_level, name='mmdet')
    
    # Log environment info
    env_info_dict = collect_env()
    env_info = '\n'.join([(f'{k}: {v}') for k, v in env_info_dict.items()])
    logger.info('Environment info:\n' + '-' * 60 + '\n' + env_info + '\n' + '-' * 60)
    
    # Set random seeds
    seed = init_random_seed(seed)
    logger.info(f'Set random seed to {seed}, deterministic: {deterministic}')
    set_random_seed(seed, deterministic=deterministic)
    cfg.seed = seed
    
    # Build model and datasets
    model = build_model(cfg.model, train_cfg=cfg.get('train_cfg'), test_cfg=cfg.get('test_cfg'))
    model.init_weights()
    datasets = [build_dataset(cfg.data.train)]
    
    # Train the model
    train_model(
        model,
        datasets,
        cfg,
        distributed=distributed,
        validate=(not no_validate),
        timestamp=timestamp,
        meta=dict(env_info=env_info, seed=seed, exp_name=osp.basename(config_path))
    )